In [1]:
"""
Примеры кода к Лекции 5.7: Middleware — декларативная устойчивость агентов

Этот модуль демонстрирует:
1. Базовое подключение middleware к агенту
2. Устойчивость: retry, fallback
3. Лимиты и контекст: ModelCallLimit, ToolCallLimit, Summarization, ContextEditing
4. Безопасность: HumanInTheLoop, PII, LLMToolSelector
5. Комбинирование нескольких middleware
6. Кастомный middleware через декоратор
7. Кастомный middleware через класс (AgentMiddleware)
"""

'\nПримеры кода к Лекции 5.7: Middleware — декларативная устойчивость агентов\n\nЭтот модуль демонстрирует:\n1. Базовое подключение middleware к агенту\n2. Устойчивость: retry, fallback\n3. Лимиты и контекст: ModelCallLimit, ToolCallLimit, Summarization, ContextEditing\n4. Безопасность: HumanInTheLoop, PII, LLMToolSelector\n5. Комбинирование нескольких middleware\n6. Кастомный middleware через декоратор\n7. Кастомный middleware через класс (AgentMiddleware)\n'

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    ContextEditingMiddleware,
    HumanInTheLoopMiddleware,
    LLMToolSelectorMiddleware,
    ModelCallLimitMiddleware,
    ModelFallbackMiddleware,
    ModelRetryMiddleware,
    PIIMiddleware,
    SummarizationMiddleware,
    ToolCallLimitMiddleware,
    ToolRetryMiddleware,
)
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from llm_config import check_api_key, get_llm

In [3]:
if not check_api_key():
    raise ValueError("API key is not set")

In [4]:
# ============================================================================
# ЧАСТЬ 1: БАЗОВОЕ ПОДКЛЮЧЕНИЕ MIDDLEWARE
# ============================================================================
def example_basic_middleware():
    """
    Базовый пример: агент с ModelRetryMiddleware и ToolCallLimitMiddleware.

    ModelRetryMiddleware автоматически повторяет вызов модели при сбоях.
    ToolCallLimitMiddleware ограничивает число вызовов инструмента.
    """
    print("\n" + "=" * 60)
    print("ПРИМЕР 1: Базовое подключение middleware")
    print("=" * 60)

    @tool
    def search(query: str) -> str:
        """Поиск информации по запросу."""
        return f"Результаты поиска по '{query}': Python 3.13 вышел в октябре 2024"

    agent = create_agent(
        model="gpt-5.4-mini",
        tools=[search],
        system_prompt="Ты помощник. Используй инструмент search для поиска информации.",
        middleware=[
            ModelRetryMiddleware(max_retries=3, initial_delay=1.0),
            ToolCallLimitMiddleware(tool_name="search", run_limit=3),
        ],
    )

    result = agent.invoke(
        {"messages": [HumanMessage(content="Когда вышел Python 3.13?")]}
    )
    print(f"  Ответ: {result['messages'][-1].content[:200]}")


if __name__ == "__main__":
    example_basic_middleware()


ПРИМЕР 1: Базовое подключение middleware


  Ответ: Python 3.13 вышел в октябре 2024 года.


In [5]:
# ============================================================================
# ЧАСТЬ 2: ВСТРОЕННЫЕ MIDDLEWARE
# ============================================================================
def example_builtin_middleware():
    """
    Демонстрация каждого встроенного middleware по отдельности.

    ModelRetryMiddleware — retry при сбоях модели.
    ToolRetryMiddleware — retry при сбоях инструментов.
    ModelFallbackMiddleware — каскадный fallback на другие модели.
    SummarizationMiddleware — сжатие длинной истории сообщений.
    """
    print("\n" + "=" * 60)
    print("ПРИМЕР 2: Встроенные middleware")
    print("=" * 60)

    @tool
    def analyze_data(query: str) -> str:
        """Анализ данных по запросу."""
        return f"Анализ '{query}': тренд положительный, рост 15% за квартал"

    # --- ModelRetryMiddleware ---
    print("\n  --- ModelRetryMiddleware ---")
    agent_retry = create_agent(
        model="gpt-5.4-mini",
        tools=[analyze_data],
        middleware=[
            ModelRetryMiddleware(
                max_retries=3,
                initial_delay=1.0,
                backoff_factor=2.0,
                on_failure="continue",  # при исчерпании попыток — AIMessage с ошибкой
            ),
        ],
    )
    result = agent_retry.invoke(
        {"messages": [HumanMessage(content="Проанализируй продажи за Q4")]}
    )
    print(f"  Ответ: {result['messages'][-1].content[:150]}")

    # --- ToolRetryMiddleware ---
    print("\n  --- ToolRetryMiddleware ---")
    agent_tool_retry = create_agent(
        model="gpt-5.4-mini",
        tools=[analyze_data],
        middleware=[
            ToolRetryMiddleware(
                max_retries=2,
                initial_delay=0.5,
                retry_on=(ConnectionError, TimeoutError),
                on_failure="return_message",  # ToolMessage с описанием ошибки
            ),
        ],
    )
    result = agent_tool_retry.invoke(
        {"messages": [HumanMessage(content="Проанализируй конверсию")]}
    )
    print(f"  Ответ: {result['messages'][-1].content[:150]}")

    # --- ModelFallbackMiddleware ---
    print("\n  --- ModelFallbackMiddleware ---")
    agent_fallback = create_agent(
        model="gpt-5.4",
        tools=[analyze_data],
        middleware=[
            ModelFallbackMiddleware("gpt-5.4-mini"),  # fallback на мини-модель
        ],
    )
    result = agent_fallback.invoke(
        {"messages": [HumanMessage(content="Краткий анализ трендов")]}
    )
    print(f"  Ответ: {result['messages'][-1].content[:150]}")

    # --- SummarizationMiddleware ---
    print("\n  --- SummarizationMiddleware ---")
    agent_summarize = create_agent(
        model="gpt-5.4-mini",
        tools=[analyze_data],
        middleware=[
            SummarizationMiddleware(
                model="gpt-5.4-mini",
                trigger=("messages", 20),  # суммаризация при >20 сообщений
                keep=("messages", 10),  # сохранить последние 10
            ),
        ],
    )
    result = agent_summarize.invoke(
        {"messages": [HumanMessage(content="Анализ за год")]}
    )
    print(f"  Ответ: {result['messages'][-1].content[:150]}")


if __name__ == "__main__":
    example_builtin_middleware()


ПРИМЕР 2: Встроенные middleware

  --- ModelRetryMiddleware ---


  Ответ: По Q4 виден **положительный тренд**: продажи **выросли на 15% за квартал**.

Чтобы сделать полноценный разбор, мне нужны сами данные или хотя бы списо

  --- ToolRetryMiddleware ---


/var/folders/hj/f38s7jd91tb80244nxj6q7f40000gn/T/ipykernel_82345/667581877.py:47: DeprecationWarning: on_failure='return_message' is deprecated and will be removed in a future version. Use on_failure='continue' instead.
  ToolRetryMiddleware(


  Ответ: Могу, но сейчас не хватает исходных данных.

Для анализа конверсии нужны хотя бы:
- период;
- количество визитов/сессий;
- количество лидов/заявок;
- 

  --- ModelFallbackMiddleware ---


  Ответ: Уточните, пожалуйста, какие именно тренды нужно кратко проанализировать:

- рынок/акции
- продажи
- соцсети
- поисковые запросы
- отрасль/технологии
-

  --- SummarizationMiddleware ---


  Ответ: По доступным данным за год:

- **Тренд:** положительный
- **Динамика:** рост примерно **15% за квартал**
- **Общий вывод:** показатели за год демонстр


In [6]:
# ============================================================================
# ЧАСТЬ 3: ЛИМИТЫ И УПРАВЛЕНИЕ КОНТЕКСТОМ
# ============================================================================
def example_limits_and_context():
    """
    ModelCallLimitMiddleware — лимит вызовов модели.
    ToolCallLimitMiddleware — лимит вызовов инструментов.
    SummarizationMiddleware — сжатие длинной истории.
    ContextEditingMiddleware — очистка старых tool results.
    """
    print("\n" + "=" * 60)
    print("ПРИМЕР 3: Лимиты и управление контекстом")
    print("=" * 60)

    @tool
    def search(query: str) -> str:
        """Поиск информации."""
        return f"Результаты: {query}"

    # ModelCallLimitMiddleware — лимит вызовов модели
    print("\n  --- ModelCallLimitMiddleware ---")
    agent = create_agent(
        model="gpt-5.4-mini",
        tools=[search],
        middleware=[
            ModelCallLimitMiddleware(
                run_limit=5,  # макс 5 вызовов модели за запуск
                exit_behavior="end",  # при превышении — завершить агента
            ),
        ],
    )
    result = agent.invoke(
        {"messages": [HumanMessage(content="Найди информацию о Python")]}
    )
    print(f"  Ответ: {result['messages'][-1].content[:100]}")

    # SummarizationMiddleware — сжатие истории
    print("\n  --- SummarizationMiddleware ---")
    agent = create_agent(
        model="gpt-5.4-mini",
        tools=[search],
        middleware=[
            SummarizationMiddleware(
                model="gpt-5.4-mini",
                trigger=("tokens", 4000),  # суммаризация при >4000 токенов
                keep=("messages", 10),
            ),
        ],
    )
    result = agent.invoke({"messages": [HumanMessage(content="Кратко о LangGraph")]})
    print(f"  Ответ: {result['messages'][-1].content[:100]}")


if __name__ == "__main__":
    example_limits_and_context()


ПРИМЕР 3: Лимиты и управление контекстом

  --- ModelCallLimitMiddleware ---


  Ответ: Python — это высокоуровневый язык программирования, который известен:

- простым и читаемым синтакси

  --- SummarizationMiddleware ---


  Ответ: LangGraph — это библиотека для построения **сложных LLM-агентов и workflows** в виде **графа состоян


In [7]:
# ============================================================================
# ЧАСТЬ 4: БЕЗОПАСНОСТЬ И КОНТРОЛЬ
# ============================================================================
def example_safety_middleware():
    """
    HumanInTheLoopMiddleware — HITL через middleware.
    PIIMiddleware — редактирование персональных данных.
    LLMToolSelectorMiddleware — отбор релевантных инструментов.
    """
    print("\n" + "=" * 60)
    print("ПРИМЕР 4: Безопасность и контроль")
    print("=" * 60)

    @tool
    def send_email(to: str, subject: str, body: str) -> str:
        """Отправляет email."""
        return f"Email отправлен на {to}"

    @tool
    def web_search(query: str) -> str:
        """Поиск в интернете."""
        return f"Результаты: {query}"

    @tool
    def calculator(expression: str) -> str:
        """Калькулятор."""
        return str(eval(expression))

    # HumanInTheLoopMiddleware — HITL без модификации кода узлов
    print("\n  --- HumanInTheLoopMiddleware ---")
    print("  send_email → требует одобрения")
    print("  web_search, calculator → автоматически")

    from langgraph.checkpoint.memory import InMemorySaver

    agent = create_agent(
        model="gpt-5.4-mini",
        tools=[send_email, web_search, calculator],
        checkpointer=InMemorySaver(),
        middleware=[
            HumanInTheLoopMiddleware(
                interrupt_on={
                    "send_email": {
                        "allowed_decisions": ["approve", "edit", "reject"],
                    },
                    "web_search": False,  # не требует одобрения
                    "calculator": False,
                }
            ),
        ],
    )
    print("  Агент создан с HITL middleware")

    # PIIMiddleware — фильтрация персональных данных
    print("\n  --- PIIMiddleware ---")
    agent_pii = create_agent(
        model="gpt-5.4-mini",
        tools=[web_search],
        middleware=[
            PIIMiddleware(
                pii_type="email",
                strategy="redact",
                apply_to_input=True,
                apply_to_output=True,
            ),
        ],
    )
    print("  Агент с PII-фильтрацией email-адресов")

    # LLMToolSelectorMiddleware — отбор релевантных инструментов
    print("\n  --- LLMToolSelectorMiddleware ---")
    agent_selector = create_agent(
        model="gpt-5.4",
        tools=[send_email, web_search, calculator],
        middleware=[
            LLMToolSelectorMiddleware(
                model="gpt-5.4-mini",
                max_tools=2,
                always_include=["web_search"],
            ),
        ],
    )
    print("  Агент с отбором инструментов (макс 2 из 3)")


if __name__ == "__main__":
    example_safety_middleware()


ПРИМЕР 4: Безопасность и контроль

  --- HumanInTheLoopMiddleware ---
  send_email → требует одобрения
  web_search, calculator → автоматически
  Агент создан с HITL middleware

  --- PIIMiddleware ---
  Агент с PII-фильтрацией email-адресов

  --- LLMToolSelectorMiddleware ---
  Агент с отбором инструментов (макс 2 из 3)


In [8]:
# ============================================================================
# ЧАСТЬ 5: КОМБИНИРОВАНИЕ MIDDLEWARE
# ============================================================================
def example_combined_middleware():
    """
    Комбинация пяти middleware для комплексной устойчивости.

    Пять строк — и агент защищён от сбоев модели, инструментов,
    недоступности провайдера, бесконечных циклов и переполнения контекста.
    """
    print("\n" + "=" * 60)
    print("ПРИМЕР 3: Комбинирование middleware")
    print("=" * 60)

    @tool
    def web_search(query: str) -> str:
        """Поиск в интернете."""
        return f"Найдено по '{query}': 5 релевантных результатов"

    @tool
    def get_report(topic: str) -> str:
        """Генерация отчёта."""
        return f"Отчёт по '{topic}': ключевые метрики стабильны"

    agent = create_agent(
        model="gpt-5.4",
        tools=[web_search, get_report],
        system_prompt="Ты аналитик. Используй инструменты для исследования.",
        middleware=[
            # 1. Retry модели: 3 попытки, backoff 1→2→4 сек
            ModelRetryMiddleware(max_retries=3, initial_delay=1.0),
            # 2. Fallback: если gpt-5.4 недоступен → gpt-5.4-mini
            ModelFallbackMiddleware("gpt-5.4-mini"),
            # 3. Retry инструментов: 2 попытки для web_search
            ToolRetryMiddleware(max_retries=2, tools=["web_search"]),
            # 4. Лимит модели: не больше 15 вызовов за запуск
            ModelCallLimitMiddleware(run_limit=15, exit_behavior="end"),
            # 5. Лимит: web_search — не больше 5 вызовов
            ToolCallLimitMiddleware(tool_name="web_search", run_limit=5),
            # 6. Суммаризация: сжатие при >4000 токенов
            SummarizationMiddleware(model="gpt-5.4-mini", trigger=("tokens", 4000)),
        ],
    )

    result = agent.invoke(
        {"messages": [HumanMessage(content="Исследуй тренды ИИ за 2025 год")]}
    )
    print(f"  Ответ: {result['messages'][-1].content[:200]}")
    print("\n  Middleware-стек:")
    print("  1. ModelRetryMiddleware — retry модели")
    print("  2. ModelFallbackMiddleware — fallback на gpt-5.4-mini")
    print("  3. ToolRetryMiddleware — retry web_search")
    print("  4. ModelCallLimitMiddleware — лимит 15 вызовов модели")
    print("  5. ToolCallLimitMiddleware — лимит 5 вызовов web_search")
    print("  6. SummarizationMiddleware — сжатие при >4000 токенов")


if __name__ == "__main__":
    example_combined_middleware()


ПРИМЕР 3: Комбинирование middleware


  Ответ: Ниже — сжатое исследование ключевых трендов ИИ в 2025 году.

## Краткий вывод
2025 год — это переход от «ажиотажа вокруг генеративного ИИ» к фазе промышленного внедрения. Главные векторы: ИИ-агенты, м

  Middleware-стек:
  1. ModelRetryMiddleware — retry модели
  2. ModelFallbackMiddleware — fallback на gpt-5.4-mini
  3. ToolRetryMiddleware — retry web_search
  4. ModelCallLimitMiddleware — лимит 15 вызовов модели
  5. ToolCallLimitMiddleware — лимит 5 вызовов web_search
  6. SummarizationMiddleware — сжатие при >4000 токенов


In [9]:
# ============================================================================
# ЧАСТЬ 4: КАСТОМНЫЙ MIDDLEWARE — ДЕКОРАТОР
# ============================================================================
def example_custom_decorator():
    """
    Кастомный middleware через декораторы.

    @before_model — проверка лимита сообщений перед вызовом модели.
    @wrap_model_call — логирование каждого вызова модели.
    """
    print("\n" + "=" * 60)
    print("ПРИМЕР 4: Кастомный middleware (декоратор)")
    print("=" * 60)

    from langchain.agents.middleware import (
        AgentState,
        ModelRequest,
        ModelResponse,
        before_model,
        wrap_model_call,
    )
    from langchain_core.messages import AIMessage

    @before_model(can_jump_to=["end"])
    def check_message_limit(state: AgentState, runtime):
        """Останавливает агента при превышении лимита сообщений."""
        if len(state["messages"]) >= 50:
            return {
                "messages": [AIMessage(content="Лимит сообщений (50) достигнут.")],
                "jump_to": "end",
            }
        return None

    @wrap_model_call
    def log_model_call(request: ModelRequest, handler):
        """Логирует каждый вызов модели."""
        print(f"    [log] Вызов модели: {len(request.messages)} сообщений")
        response = handler(request)
        last_msg = response.result[-1] if response.result else None
        content = last_msg.content[:60] if last_msg and last_msg.content else ""
        print(f"    [log] Ответ: {content}...")
        return response

    @tool
    def calculator(expression: str) -> str:
        """Вычисляет математическое выражение."""
        try:
            return str(eval(expression))
        except Exception as e:
            return f"Ошибка: {e}"

    agent = create_agent(
        model="gpt-5.4-mini",
        tools=[calculator],
        middleware=[check_message_limit, log_model_call],
    )

    result = agent.invoke(
        {"messages": [HumanMessage(content="Сколько будет 2 ** 10?")]}
    )
    print(f"\n  Ответ: {result['messages'][-1].content}")


if __name__ == "__main__":
    example_custom_decorator()


ПРИМЕР 4: Кастомный middleware (декоратор)
    [log] Вызов модели: 1 сообщений


    [log] Ответ: ...
    [log] Вызов модели: 3 сообщений


    [log] Ответ: 2 ** 10 = 1024...

  Ответ: 2 ** 10 = 1024


In [10]:
# ============================================================================
# ЧАСТЬ 5: КАСТОМНЫЙ MIDDLEWARE — КЛАСС
# ============================================================================
def example_custom_class():
    """
    Кастомный middleware через класс AgentMiddleware.

    Класс позволяет хранить внутреннее состояние (счётчики, кэши)
    и параметризовать middleware при создании.
    """
    print("\n" + "=" * 60)
    print("ПРИМЕР 5: Кастомный middleware (класс)")
    print("=" * 60)

    from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
    from langchain_core.messages import AIMessage

    class CallCounterMiddleware(AgentMiddleware):
        """Считает вызовы модели и останавливает при превышении лимита."""

        def __init__(self, max_calls: int = 10):
            super().__init__()
            self.max_calls = max_calls
            self.call_count = 0

        @hook_config(can_jump_to=["end"])
        def before_model(self, state: AgentState, runtime):
            self.call_count += 1
            print(f"    [counter] Вызов модели #{self.call_count}/{self.max_calls}")
            if self.call_count > self.max_calls:
                return {
                    "messages": [
                        AIMessage(
                            content=f"Лимит вызовов модели ({self.max_calls}) исчерпан."
                        )
                    ],
                    "jump_to": "end",
                }
            return None

        def after_model(self, state: AgentState, runtime):
            last_msg = state["messages"][-1]
            content = last_msg.content[:50] if last_msg.content else "(пусто)"
            print(f"    [counter] Ответ модели: {content}...")
            return None

    @tool
    def get_weather(city: str) -> str:
        """Получить погоду в городе."""
        return f"В {city} сейчас +22°C, облачно"

    counter = CallCounterMiddleware(max_calls=5)

    agent = create_agent(
        model="gpt-5.4-mini",
        tools=[get_weather],
        middleware=[counter],
    )

    result = agent.invoke(
        {"messages": [HumanMessage(content="Какая погода в Москве?")]}
    )
    print(f"\n  Ответ: {result['messages'][-1].content}")
    print(f"  Всего вызовов модели: {counter.call_count}")


if __name__ == "__main__":
    example_custom_class()


ПРИМЕР 5: Кастомный middleware (класс)
    [counter] Вызов модели #1/5


    [counter] Ответ модели: (пусто)...
    [counter] Вызов модели #2/5


    [counter] Ответ модели: В Москве сейчас +22°C, облачно....

  Ответ: В Москве сейчас +22°C, облачно.
  Всего вызовов модели: 2
